In [1]:
import pandas as pd
import numpy as np

fund_master = pd.read_csv("../data/raw/01_fund_master.csv")
nav = pd.read_csv("../data/raw/02_nav_history.csv")
aum = pd.read_csv("../data/raw/03_aum_by_fund_house.csv")
sip = pd.read_csv("../data/raw/04_monthly_sip_inflows.csv")
category = pd.read_csv("../data/raw/05_category_inflows.csv")
folios = pd.read_csv("../data/raw/06_industry_folio_count.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")
transactions = pd.read_csv("../data/raw/08_investor_transactions.csv")
holdings = pd.read_csv("../data/raw/09_portfolio_holdings.csv")
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")

In [2]:
nav["date"] = pd.to_datetime(nav["date"])

In [3]:
nav = nav.sort_values(
    ["amfi_code","date"]
)

In [4]:
nav = nav.drop_duplicates()

In [5]:
nav["nav"] = (
    nav.groupby("amfi_code")["nav"]
       .ffill()
)

In [6]:
invalid_nav = nav[nav["nav"] <= 0]

print("Invalid NAV rows:", len(invalid_nav))

Invalid NAV rows: 0


In [7]:
nav = nav[nav["nav"] > 0]

In [8]:
nav.to_csv(
    "../data/processed/02_nav_history_clean.csv",
    index=False
)

In [9]:
transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"]
)

In [10]:
transactions["transaction_type"].unique()
transactions["transaction_type"] = (
    transactions["transaction_type"]
    .str.strip()
    .str.title()
)


In [11]:
transactions = transactions[
    transactions["amount_inr"] > 0
]
transactions["kyc_status"].value_counts()


kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64

In [12]:
valid = [
    "Verified",
    "Pending",
    "Rejected"
]

bad_kyc = transactions[
    ~transactions["kyc_status"].isin(valid)
]

print(len(bad_kyc))

0


In [13]:
transactions.to_csv(
    "../data/processed/08_investor_transactions_clean.csv",
    index=False
)

In [14]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio"
]
for col in return_cols:
    
    performance[col] = pd.to_numeric(
        performance[col],
        errors="coerce"
    )


In [15]:
performance[
    return_cols
].isnull().sum()
anomaly = performance[
    (performance["expense_ratio_pct"] < 0.1)
    |
    (performance["expense_ratio_pct"] > 2.5)
]

print(anomaly)

Empty DataFrame
Columns: [amfi_code, scheme_name, fund_house, category, plan, return_1yr_pct, return_3yr_pct, return_5yr_pct, benchmark_3yr_pct, alpha, beta, sharpe_ratio, sortino_ratio, std_dev_ann_pct, max_drawdown_pct, aum_crore, expense_ratio_pct, morningstar_rating, risk_grade]
Index: []


In [16]:
performance.to_csv(
    "../data/processed/07_scheme_performance_clean.csv",
    index=False
)

In [17]:
fund_master.to_csv("../data/processed/01_fund_master_clean.csv",index=False)
aum.to_csv("../data/processed/03_aum_by_fund_house_clean.csv",index=False)
sip.to_csv("../data/processed/04_monthly_sip_inflows_clean.csv",index=False)
category.to_csv("../data/processed/05_category_inflows_clean.csv",index=False)
folios.to_csv("../data/processed/06_industry_folio_count_clean.csv",index=False)
holdings.to_csv("../data/processed/09_portfolio_holdings_clean.csv",index=False)
benchmark.to_csv("../data/processed/10_benchmark_indices_clean.csv",index=False)

In [18]:
from sqlalchemy import create_engine

engine = create_engine(
    "sqlite:///../database/bluestock_mf.db"
)

In [1]:
fund_master.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

nav.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

transactions.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

performance.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

aum.to_sql(
    "fact_aum",
    engine,
    if_exists="replace",
    index=False
)
scorecard["fund_score"] = (
    scorecard["return_rank"] * 30 +
    scorecard["sharpe_rank"] * 25 +
    scorecard["alpha_rank"] * 20 +
    scorecard["expense_rank"] * 15 +
    scorecard["dd_rank"] * 10
)
scorecard.to_csv(
    "../reports/fund_scorecard.csv",
    index=False
)

NameError: name 'fund_master' is not defined

In [2]:
print(scorecard.columns.tolist())

NameError: name 'scorecard' is not defined